# Введение в прикладной ИИ
## Python для данных: NumPy, pandas и разведочный анализ (EDA) на примере «Титаника»

**Библиотеки:** `numpy`, `pandas`, `matplotlib`, `scikit-learn` — в Colab уже установлены; локально: `pip install numpy pandas matplotlib scikit-learn`.

**Данные:** `titanic.csv` (train-часть соревнования Kaggle Titanic, 891 пассажир). Положите файл рядом с ноутбуком — или ячейка загрузки скачает его сама.

## 1. Где мы находимся: пайплайн прикладного ИИ

Любой прикладной ML/DL-проект устроен одинаково — табличка это, картинки или тексты:

```
данные → загрузка → EDA (разведка) → очистка и признаки → модель → метрики → выводы / продакшн
        └──────────────── ЛР2 ────────────────┘            └──── ЛР3 (ML), ЛР4–5 (DL) ────┘
```

По опросам практиков **60–80 % времени** проекта уходит на левую часть схемы: понять, почистить и подготовить данные. Сама модель — это несколько строк `fit()` / `predict()`. Поэтому курс по ИИ начинается не с нейросетей, а с NumPy и pandas.

**Три уровня, на которых живут данные в Python**

| Уровень | Инструмент | Что умеет | Когда использовать |
|---|---|---|---|
| объекты Python | `list`, `dict`, циклы | гибко, но медленно | мелкая логика, парсинг, «склейка» |
| массивы | **NumPy** `ndarray` | быстрые числовые операции над векторами и матрицами | математика, модели |
| таблицы | **pandas** `DataFrame` | именованные столбцы разных типов, пропуски, группировки, чтение файлов | загрузка и разведка данных |

В основании — NumPy: pandas хранит столбцы в NumPy-массивах, scikit-learn принимает NumPy-матрицы, тензоры PyTorch конвертируются в NumPy одной командой.

### Задача «Титаник»

В ночь на 15 апреля 1912 года «Титаник» столкнулся с айсбергом; из 2224 человек на борту погибло более 1500. Шлюпок не хватало, и выживание зависело не только от удачи: действовало правило «женщины и дети вперёд», а каюты 1-го класса находились ближе к шлюпочной палубе.

**Постановка.** По данным о пассажире (пол, возраст, класс билета, семья на борту, цена билета, порт посадки…) предсказать, выжил он или нет.

- Это **бинарная классификация**: целевая переменная `Survived ∈ {0, 1}`.
- Данные: 891 пассажир — train-часть соревнования Kaggle «Titanic: Machine Learning from Disaster».
- Почему именно он: маленький, понятный без предметных знаний и при этом содержит все типичные «болячки» реальных данных — пропуски, категориальные и текстовые признаки, выбросы, дисбаланс классов.

По такому же чеклисту вы будете делать ЛР2 на своём датасете.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Настройки отображения таблиц
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
pd.set_option('display.precision', 3)

# Палитра и стиль графиков — один раз на весь ноутбук
BLUE, ORANGE, AQUA, GRAY = '#2a78d6', '#eb6834', '#1baf7a', '#898781'
plt.rcParams.update({
    'figure.figsize': (7, 4),
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.linestyle': '--', 'grid.alpha': 0.4, 'axes.axisbelow': True,
    'axes.titleweight': 'bold', 'axes.titlesize': 12,
})

print('Python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__)

## 2. Python-минимум для анализа данных

Из всего языка в ноутбуках аналитика и ML-инженера ежедневно используется небольшое подмножество: списки, словари, comprehension'ы, f-строки, функции, `lambda`, срезы, `enumerate`/`zip`. Проверим себя на нём и заодно посмотрим, как выглядят «сырые» данные *до* pandas.

Сырые данные часто приходят как **список словарей** — ровно так выглядит JSON-ответ любого API.

In [ ]:
passengers = [
    {"name": "Braund, Mr. Owen Harris",    "sex": "male",   "age": 22,   "pclass": 3, "survived": 0},
    {"name": "Cumings, Mrs. John Bradley", "sex": "female", "age": 38,   "pclass": 1, "survived": 1},
    {"name": "Heikkinen, Miss. Laina",     "sex": "female", "age": 26,   "pclass": 3, "survived": 1},
    {"name": "Allen, Mr. William Henry",   "sex": "male",   "age": 35,   "pclass": 3, "survived": 0},
    {"name": "Moran, Mr. James",           "sex": "male",   "age": None, "pclass": 3, "survived": 0},  # возраст неизвестен
]

first = passengers[0]
print(type(passengers).__name__, len(passengers))
print(type(first).__name__, '|', first["name"], '|', first.get("cabin", "нет данных"))  # .get не падает на отсутствующем ключе

In [ ]:
# List comprehension — «цикл в одну строку», основной инструмент преобразования данных
ages = [p["age"] for p in passengers if p["age"] is not None]
names_upper = [p["name"].upper() for p in passengers]

# Dict comprehension: имя -> выжил
survived_by_name = {p["name"]: p["survived"] for p in passengers}

# Генератор + sum/len: доля выживших среди женщин
women = [p for p in passengers if p["sex"] == "female"]
rate = sum(p["survived"] for p in women) / len(women)

print(ages)
print(names_upper[:2])
print(survived_by_name)
print(f"Доля выживших женщин: {rate:.1%}")          # f-строка с форматом: .1% -> проценты с одним знаком

In [ ]:
# Сортировка по ключу; lambda — безымянная функция из одного выражения
by_age = sorted([p for p in passengers if p["age"] is not None], key=lambda p: p["age"], reverse=True)
for i, p in enumerate(by_age, start=1):             # enumerate даёт номер и элемент
    print(i, p["age"], p["name"])

# zip — идти по двум последовательностям параллельно
sexes = [p["sex"] for p in passengers]
targets = [p["survived"] for p in passengers]
for sex, target in zip(sexes, targets):
    print(sex, "->", target, end="   ")
print()

mean_age, n = sum(ages) / len(ages), len(ages)     # распаковка кортежа в две переменные
print(f"Средний возраст по {n} известным значениям: {mean_age:.2f}")

# Срезы одинаковы для строк и списков — и дальше для массивов NumPy и таблиц pandas
print(sexes[1:3], sexes[-1], sexes[::2])

In [ ]:
def survival_rate(rows, key, value):
    '''Доля выживших среди записей, у которых rows[i][key] == value.'''
    group = [r for r in rows if r[key] == value]
    if not group:
        return float("nan")                 # «не число» — так в данных обозначают отсутствие значения
    return sum(r["survived"] for r in group) / len(group)

for sex in ("male", "female"):
    print(f"{sex:>7}: {survival_rate(passengers, 'sex', sex):.2f}")

Три вещи, на которые стоит обратить внимание:

1. **`None` → `nan`.** Пропуск в данных — не ошибка, а нормальное состояние. Дальше увидим, как NumPy и pandas с ним работают.
2. Логика «разбить по группам и посчитать среднее» заняла 6 строк. В pandas это будет **одна**: `df.groupby('Sex')['Survived'].mean()`.
3. На 5 записях всё быстро. А на 5 миллионах? Проверим.

## 3. NumPy — быстрые массивы

### Зачем, если есть списки?

Список Python — это массив **указателей** на объекты. Каждое число `int` — отдельный объект в памяти (28 байт на число `5`!), а цикл `for` на каждом шаге проходит через интерпретатор: проверить тип, найти метод `__add__`, создать новый объект результата.

`np.ndarray` — **непрерывный блок памяти одного типа** (`dtype`) плюс метаданные (`shape`, `strides`). Операция `a + b` — один вызов скомпилированного C-цикла, без интерпретатора на каждом элементе. Отсюда ускорение в десятки раз и экономия памяти в разы.

Это не «оптимизация ради оптимизации»: обучение модели — миллиарды умножений. Нейросеть на чистом Python не обучилась бы никогда.

In [ ]:
n = 1_000_000
lst = list(range(n))          # список Python
arr = np.arange(n)            # массив NumPy

list_bytes = sys.getsizeof(lst) + sum(sys.getsizeof(v) for v in lst)
print(f"list:    {list_bytes / 1e6:5.1f} МБ  (массив указателей + отдельный объект int на каждое число)")
print(f"ndarray: {arr.nbytes / 1e6:5.1f} МБ  ({arr.itemsize} байт × {n} элементов, dtype={arr.dtype})")

In [ ]:
# Медленно vs быстро: сумма квадратов миллиона чисел
%timeit -r 3 sum(x * x for x in lst)      # чистый Python: цикл в интерпретаторе
%timeit -r 3 (arr * arr).sum()            # NumPy: векторизованно
%timeit -r 3 arr @ arr                    # NumPy: скалярное произведение одним вызовом

Правило на весь курс: **если вы пишете `for` по элементам массива — скорее всего, есть векторизованная операция, которая сделает то же в 10–100 раз быстрее.**

### Создание массивов

In [ ]:
a = np.array([1, 2, 3, 4])                 # из списка; dtype подбирается автоматически
b = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])            # вложенные списки -> двумерный массив (матрица)

print(a.dtype, a.shape, a.ndim)
print(b.dtype, b.shape, b.ndim)

print(np.zeros((2, 3)))                    # нули заданной формы
print(np.ones(3, dtype=int))               # единицы, целые
print(np.arange(0, 10, 2.5))               # как range, но шаг может быть дробным
print(np.linspace(0, 1, 5))                # 5 точек равномерно на отрезке [0, 1]

In [ ]:
rng = np.random.default_rng(seed=42)       # генератор с фиксированным зерном -> воспроизводимость (в ML обязательно)
print(rng.integers(0, 10, size=5))         # целые из [0, 10)
print(rng.normal(loc=0, scale=1, size=(2, 3)).round(2))   # нормальное распределение
print(rng.choice(["S", "C", "Q"], size=5)) # случайный выбор из списка

### Индексация, срезы, маски

In [ ]:
x = np.array([10, 20, 30, 40, 50, 60])

print(x[0], x[-1], x[1:4], x[::2])         # как у списков

mask = x > 25                              # булева маска — главный инструмент фильтрации
print(mask)
print(x[mask])
print(x[(x > 15) & (x < 55)])              # & и | вместо and/or, и обязательно скобки

print(x[[0, 2, 5]])                        # fancy indexing — выбрать по списку позиций

In [ ]:
# Под капотом: срез — это «окно» в ту же память (view), а не копия
y = x[1:3]
y[0] = 999
print(x)                                   # исходный массив изменился!

z = x[1:3].copy()                          # явная копия, если нужна независимость
z[0] = -1
print(x)

### Матрица «объекты × признаки» и ось `axis`

В ML данные — это матрица $X$ формы `(n_samples, n_features)`: **строка = объект** (пассажир), **столбец = признак** (возраст, цена билета, …). Вектор $y$ длины `n_samples` — целевая переменная. Все библиотеки (scikit-learn, PyTorch) ждут именно такую форму.

Параметр `axis` говорит, **вдоль какой оси схлопывать**:
- `axis=0` — идём вниз по строкам, результат на каждый **столбец** (статистика по признаку);
- `axis=1` — идём вправо по столбцам, результат на каждую **строку** (статистика по объекту).

In [ ]:
# 4 пассажира × 3 признака: [возраст, цена билета, размер семьи]
X = np.array([[22.0,  7.25, 2],
              [38.0, 71.28, 2],
              [26.0,  7.92, 1],
              [35.0, 53.10, 2]])
y = np.array([0, 1, 1, 1])

print("shape:", X.shape, "-> объектов:", X.shape[0], ", признаков:", X.shape[1])
print("X[1, 0] =", X[1, 0], "| строка X[1] =", X[1], "| столбец X[:, 1] =", X[:, 1])
print("среднее по признакам (axis=0):", X.mean(axis=0).round(2))
print("сумма по объектам    (axis=1):", X.sum(axis=1).round(2))
print("общее среднее:", X.mean().round(2), "| индекс максимума цены:", X[:, 1].argmax())

### Broadcasting — операции между массивами разной формы

NumPy «растягивает» меньший массив вдоль недостающих осей, если формы совместимы (размеры равны или один из них равен 1). Самый частый случай в ML — **стандартизация признаков**: из каждого столбца вычесть его среднее и поделить на стандартное отклонение:

$$x'_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

Именно это делает `StandardScaler` из scikit-learn. Без этого линейные модели и нейросети учатся плохо: признаки разного масштаба (возраст 0–80, цена 0–500) «перетягивают» веса.

In [ ]:
mu = X.mean(axis=0)           # shape (3,)
sigma = X.std(axis=0)         # shape (3,)
X_scaled = (X - mu) / sigma   # (4,3) - (3,): вектор растянулся на все 4 строки

print("до:    средние =", X.mean(axis=0).round(2), " std =", X.std(axis=0).round(2))
print("после: средние =", X_scaled.mean(axis=0).round(2), " std =", X_scaled.std(axis=0).round(2))

print(X * 2)                                            # скаляр растягивается на всё
print(X + np.array([[100], [200], [300], [400]]))        # (4,3) + (4,1): столбец растянулся по строкам

In [ ]:
fare = X[:, 1]
print(np.log1p(fare).round(2))                      # log(1 + x): «сжимает» длинный хвост распределения — пригодится в EDA
print(np.where(fare > 30, "дорого", "дёшево"))      # векторный if-else
print(np.clip(fare, 0, 50))                         # обрезать значения в диапазон — борьба с выбросами

# NaN — специальное число с плавающей точкой: «значения нет»
age = np.array([22.0, np.nan, 26.0, 35.0])
print(np.nan == np.nan)                             # False! NaN не равен даже самому себе
print(np.isnan(age), age.mean(), np.nanmean(age))   # mean «заражается» NaN, nanmean игнорирует пропуски

In [ ]:
# Линейная модель — это одно матричное умножение: y_hat = X @ w + b
w = np.array([-0.2, 1.0, 0.1])        # веса признаков (пока взяты с потолка — их подбирает обучение)
b = 0.0
logits = X_scaled @ w + b             # (4,3) @ (3,) -> (4,)
proba = 1 / (1 + np.exp(-logits))     # сигмоида -> вероятность класса 1 (так устроена логистическая регрессия)
y_pred = (proba > 0.5).astype(int)    # порог 0.5

print("вероятности: ", proba.round(3))
print("предсказания:", y_pred, " истина:", y)
print("accuracy:", (y_pred == y).mean())    # сравнение массивов -> булев массив -> среднее = доля верных
print("X.T:", X.T.shape, " X.T @ X:", (X.T @ X).shape)   # транспонирование и матричное произведение

In [ ]:
v = np.arange(6)
print(v.reshape(2, 3))                              # та же память, другая форма
print(v.reshape(-1, 1).shape)                       # -1 = «посчитай сам»; столбец (6,1) — так sklearn хочет один признак
print(np.concatenate([X, X[:1]], axis=0).shape)     # добавить строки
print(np.column_stack([X, y]).shape)                # добавить столбец (признаки + целевая)

### NumPy: что запомнить

| Задача | Как |
|---|---|
| создать | `np.array`, `np.zeros`, `np.ones`, `np.arange`, `np.linspace`, `rng.normal` |
| посмотреть | `.shape`, `.dtype`, `.ndim` |
| выбрать | `x[i]`, `x[a:b]`, `x[mask]`, `X[:, j]`, `x[[i, j]]` |
| посчитать по оси | `.mean(axis=0)`, `.sum(axis=1)`, `.max()`, `.argmax()` |
| поэлементно | `+ - * /`, `np.log1p`, `np.exp`, `np.where`, `np.clip` |
| матрицы | `X @ w`, `X.T`, `np.linalg.inv / norm / solve` |
| пропуски | `np.nan`, `np.isnan`, `np.nanmean` |

Срез — представление (view), а не копия. Цикл `for` по элементам — почти всегда признак того, что есть векторизованная замена.

## 4. pandas — таблицы с именами

NumPy хорош для чисел одной природы. Но реальные данные — **таблица**: столбцы разных типов (числа, строки, даты), с именами, с пропусками. Для этого есть pandas.

**Под капотом:** `DataFrame` — словарь столбцов, где каждый столбец (`Series`) — NumPy-массив одного типа, а у всех столбцов общий **индекс** (подписи строк). Поэтому всё, что мы знаем про NumPy (векторизация, маски, `axis`), работает и здесь, а `df.to_numpy()` отдаёт матрицу для модели.

In [ ]:
s = pd.Series([22, 38, 26, 35], index=["Braund", "Cumings", "Heikkinen", "Allen"], name="age")
print(s)
print(s["Cumings"], s.mean(), s.to_numpy())    # доступ по метке; статистики; NumPy-массив под капотом

df_small = pd.DataFrame(passengers)            # список словарей -> таблица; None превратился в NaN
df_small

### Загрузка данных

`pd.read_csv` читает CSV/TSV (параметры `sep`, `encoding`, `parse_dates`, `index_col`). Есть также `read_excel`, `read_json`, `read_sql`, `read_parquet`. Функция ниже ищет файл рядом с ноутбуком, а если его нет — скачивает с GitHub.

In [ ]:
from pathlib import Path

TITANIC_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

def load_titanic() -> pd.DataFrame:
    for candidate in ["titanic.csv", "data/titanic.csv", "train.csv", "/content/train.csv"]:
        if Path(candidate).exists():
            print("читаю", candidate)
            return pd.read_csv(candidate)
    print("локального файла нет — скачиваю", TITANIC_URL)
    return pd.read_csv(TITANIC_URL)

df = load_titanic()
print(df.shape)        # (строк, столбцов)
df.head()              # первые 5 строк

In [ ]:
df.sample(5, random_state=42)   # случайные 5 строк — честнее, чем head(): файл может быть отсортирован

### Словарь данных (data dictionary)

Первое, что делают с новым датасетом, — читают описание столбцов. Без него EDA превращается в гадание.

| Столбец | Что это | Значения |
|---|---|---|
| `PassengerId` | идентификатор | уникальный номер |
| **`Survived`** | **целевая переменная** | 0 — погиб, 1 — выжил |
| `Pclass` | класс билета (≈ социальный статус) | 1, 2, 3 |
| `Name` | имя с титулом | текст |
| `Sex` | пол | male / female |
| `Age` | возраст, лет | число, есть пропуски |
| `SibSp` | братьев/сестёр/супругов на борту | целое |
| `Parch` | родителей/детей на борту | целое |
| `Ticket` | номер билета | текст |
| `Fare` | цена билета, £ | число |
| `Cabin` | номер каюты | текст, много пропусков |
| `Embarked` | порт посадки | C — Cherbourg, Q — Queenstown, S — Southampton |

In [ ]:
df.info()     # типы, число непустых значений, память — одной командой

> В pandas 2.x текстовые столбцы показываются как `object`, в pandas 3 — как `str`. Смысл один: «строка».

In [ ]:
df.describe()                    # статистики по числовым столбцам: count, mean, std, min, квартили, max

In [ ]:
df.describe(exclude='number')    # по нечисловым: сколько уникальных значений, самое частое и его частота

### Выбор столбцов и строк

- `df['col']` → `Series`; `df[['a', 'b']]` → `DataFrame`.
- `.loc[метки строк, метки столбцов]` — по **подписям**; `.iloc[позиции, позиции]` — по **номерам**.
- Булева маска в квадратных скобках — фильтр строк.

In [ ]:
print(type(df['Age']).__name__, type(df[['Name', 'Age']]).__name__)

df.loc[0:2, ['Name', 'Sex', 'Age']]     # .loc: по меткам; обратите внимание — 0:2 включает 2!

In [ ]:
df.iloc[0:3, 3:6]                       # .iloc: по позициям, как обычные срезы (3 не включается)

In [ ]:
matrix = df[['Pclass', 'Age', 'Fare']].to_numpy()    # таблица -> NumPy-матрица (n_samples, n_features): так данные уходят в модель
print(type(matrix).__name__, matrix.shape, matrix.dtype)
matrix[:3]

In [ ]:
children = df[df['Age'] < 10]
print(len(children), "детей младше 10 лет")

rich_women = df[(df['Sex'] == 'female') & (df['Fare'] > 100)]      # & | ~ и скобки — как в NumPy
print(len(rich_women), "женщин с билетом дороже 100 £")

print(len(df[df['Embarked'].isin(['C', 'Q'])]), "сели не в Саутгемптоне")
print(len(df.query("Age >= 60 and Pclass == 1")), "пассажиров 60+ в 1-м классе")   # query — тот же фильтр, но строкой

In [ ]:
df.sort_values('Fare', ascending=False)[['Name', 'Pclass', 'Fare']].head(3)   # сортировка + выбор столбцов

In [ ]:
print(df['Pclass'].value_counts())                                          # сколько раз встречается каждое значение
print(df['Embarked'].value_counts(normalize=True, dropna=False).round(3))   # доли, с учётом пропусков
print(df['Sex'].unique().tolist(), '| уникальных билетов:', df['Ticket'].nunique())   # уникальные значения; их число

### Новые столбцы

Арифметика между столбцами векторизована (это NumPy под капотом). Для сопоставления «значение → значение» есть `.map(dict)`, для строк — аксессор `.str`, для произвольной функции — `.apply` (медленно, используйте в последнюю очередь).

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1                   # векторная арифметика
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)                 # булев столбец -> 0/1
df['SexCode'] = df['Sex'].map({'male': 0, 'female': 1})             # кодирование словарём
df['AgeGroup'] = np.where(df['Age'] < 18, 'child', 'adult')         # векторный if-else
df['Surname'] = df['Name'].str.split(',').str[0]                    # строковые методы ко всему столбцу
df['FareRounded'] = df['Fare'].apply(lambda f: round(f, -1))        # apply: функция к каждому элементу

df[['Name', 'Surname', 'FamilySize', 'IsAlone', 'SexCode', 'AgeGroup', 'FareRounded']].head()

> Ловушка: `np.where(df['Age'] < 18, ...)` для пропущенного возраста даёт `False` → такие пассажиры попали в `adult`. Любое сравнение с `NaN` — `False`. Сначала разбираемся с пропусками, потом строим признаки.

In [ ]:
# Медленно vs быстро, версия pandas: apply по строкам против векторной операции
%timeit -r 3 df.apply(lambda row: row['SibSp'] + row['Parch'] + 1, axis=1)
%timeit -r 3 df['SibSp'] + df['Parch'] + 1

### Пропуски

`NaN` — «значения нет». Он не равен ничему, не участвует в `==`, но корректно игнорируется агрегатами (`mean`, `sum`, `median`). Инструменты:

- найти: `isna()`, `notna()`; посчитать: `isna().sum()`, `isna().mean()`;
- убрать: `dropna()` (строки или столбцы), `drop(columns=...)`;
- заполнить: `fillna(значение)`, `fillna(df['col'].median())`, по группам — `groupby(...).transform(...)`.

**Чем заполнять — это решение, а не техника.** Его принимают на этапе EDA, глядя на то, *почему* пропуск возник.

In [ ]:
missing = pd.DataFrame({
    'count': df.isna().sum(),
    'share': df.isna().mean().round(3),
}).query('count > 0').sort_values('count', ascending=False)
missing

In [ ]:
demo = df[['Name', 'Age', 'Embarked']].head(6).copy()
print(len(demo), '->', len(demo.dropna(subset=['Age'])), 'строк после dropna по Age')   # строка с NaN выброшена
demo['Age_filled'] = demo['Age'].fillna(df['Age'].median())    # а здесь NaN заполнен медианой по всей таблице
demo

### Группировки: split → apply → combine

`groupby` разбивает таблицу на группы по значениям столбца, применяет агрегат к каждой группе и собирает результат. Это главный инструмент EDA: «как целевая переменная зависит от …?»

In [ ]:
df.groupby('Sex')['Survived'].mean()        # доля выживших по полу (среднее от 0/1 — это и есть доля)

In [ ]:
df.groupby(['Pclass', 'Sex'])['Survived'].agg(['mean', 'count']).round(3)   # два ключа группировки, два агрегата

In [ ]:
(df.groupby('Pclass')
   .agg(passengers=('PassengerId', 'count'),     # именованные агрегаты: имя=(столбец, функция)
        survival_rate=('Survived', 'mean'),
        median_age=('Age', 'median'),
        mean_fare=('Fare', 'mean'))
   .round(2))

In [ ]:
pd.pivot_table(df, values='Survived', index='Pclass', columns='Sex', aggfunc='mean').round(2)   # сводная таблица

In [ ]:
pd.crosstab(df['Pclass'], df['Survived'], margins=True)    # таблица сопряжённости: сколько кого

In [ ]:
age_bins = pd.cut(df['Age'], bins=[0, 12, 18, 30, 45, 60, 100])        # числовой признак -> интервалы
df.groupby(age_bins, observed=True)['Survived'].agg(['mean', 'count']).round(3)

In [ ]:
ports = pd.DataFrame({'Embarked': ['C', 'Q', 'S'],
                      'Port': ['Cherbourg', 'Queenstown', 'Southampton']})
df = df.merge(ports, on='Embarked', how='left')       # как LEFT JOIN в SQL: расшифровали коды
df[['Name', 'Embarked', 'Port']].head(3)

In [ ]:
# Цепочка методов читается сверху вниз, как SQL-запрос
report = (df[df['Age'].notna()]
            .assign(AgeBin=lambda d: pd.cut(d['Age'], [0, 18, 60, 100], labels=['child', 'adult', 'senior']))
            .groupby(['AgeBin', 'Sex'], observed=True)['Survived']
            .mean()
            .unstack()                  # второй уровень индекса -> в столбцы
            .round(2))
report.to_csv('report_survival.csv')    # сохранить результат; index=False, если индекс не нужен
report

### pandas: что запомнить

| Задача | Как |
|---|---|
| загрузить / сохранить | `pd.read_csv`, `df.to_csv(index=False)` |
| осмотреть | `head`, `sample`, `shape`, `info()`, `describe()` |
| выбрать | `df['a']`, `df[['a','b']]`, `.loc[метки]`, `.iloc[позиции]`, `df[маска]`, `query` |
| новый столбец | арифметика, `.map(dict)`, `.str.*`, `np.where`, `.apply` (в последнюю очередь) |
| пропуски | `isna().sum()`, `dropna`, `fillna`, `groupby().transform` |
| агрегировать | `value_counts`, `groupby().agg()`, `pivot_table`, `crosstab`, `cut` / `qcut` |
| соединить | `merge` (JOIN), `concat` (склеить) |
| в модель | `df.to_numpy()`, `df[cols].to_numpy()` |

## 5. EDA «Титаника»: разведочный анализ по чеклисту

**EDA (Exploratory Data Analysis)** — систематический осмотр данных *до* моделирования. Нужно ответить на вопросы:

1. Что за задача и что за целевая переменная? Сбалансированы ли классы?
2. Какие признаки есть, какого они типа, в каком масштабе?
3. Где пропуски, почему они возникли и что с ними делать?
4. Как распределён каждый признак (смещение, выбросы)?
5. Как признаки связаны с целевой переменной? А друг с другом?
6. Какие новые признаки можно построить?

Это ровно пункты ЛР2. Пройдём их на «Титанике».

### 5.1 Целевая переменная

In [ ]:
df = load_titanic()                       # начнём с чистой таблицы

target_counts = df['Survived'].value_counts().sort_index()
target_share = df['Survived'].value_counts(normalize=True).sort_index()
print(target_counts.to_dict(), target_share.round(3).to_dict())

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(['Погиб (0)', 'Выжил (1)'], target_counts.values, color=[BLUE, ORANGE], width=0.6)
ax.bar_label(bars, labels=[f"{v} ({s:.0%})" for v, s in zip(target_counts, target_share)], padding=3)
ax.set_title('Распределение целевой переменной')
ax.set_ylabel('пассажиров')
ax.set_ylim(0, 650)
plt.show()

**Вывод.** Классы несбалансированы: 62 % погибших против 38 % выживших. Следствие для ML: модель, которая *всегда* отвечает «погиб», получит accuracy ≈ 0.62, ничего не зная о пассажирах. Это **нижняя планка (baseline)** — модель с accuracy ниже бесполезна. При сильном дисбалансе (95/5) accuracy вообще перестаёт быть информативной — нужны precision, recall, ROC-AUC (ЛР3).

### 5.2 Какие признаки есть и какого они типа

Тип хранения (`int64`, `float64`, `object`) — не то же самое, что **смысловой тип** признака. `Pclass` хранится числом, но это упорядоченная категория; `PassengerId` — число, но это просто идентификатор.

| Смысловой тип | Признаки Titanic | Что с ним делать |
|---|---|---|
| числовой непрерывный | `Age`, `Fare` | распределение, выбросы, масштабирование |
| числовой счётный | `SibSp`, `Parch` | часто превращают в категории или агрегаты |
| категориальный | `Sex`, `Embarked` | кодирование (0/1, one-hot) |
| порядковый | `Pclass` | можно оставить числом — порядок есть |
| текстовый | `Name`, `Ticket`, `Cabin` | извлечь признаки регуляркой или удалить |
| идентификатор | `PassengerId` | удалить — не несёт информации |

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
text_cols = df.select_dtypes(exclude='number').columns.tolist()
print("числовые: ", numeric_cols)
print("текстовые:", text_cols)
print("\nуникальных значений в столбце:")
print(df.nunique().sort_values())

`nunique()` подсказывает смысловой тип: 2–3 уникальных значения — категория (`Sex`, `Pclass`, `Embarked`); 681 билет на 891 пассажира — почти идентификатор; `Name` уникально у каждого, но внутри есть титул — его можно извлечь.

### 5.3 Пропуски: где, почему и что делать

In [ ]:
missing_share = df.isna().mean().sort_values(ascending=False)
missing_share = missing_share[missing_share > 0]

fig, ax = plt.subplots(figsize=(6, 2.8))
bars = ax.barh(missing_share.index, missing_share.values, color=BLUE, height=0.5)
ax.bar_label(bars, labels=[f"{v:.1%}" for v in missing_share.values], padding=3)
ax.set_xlim(0, 1)
ax.set_title('Доля пропусков по столбцам')
ax.invert_yaxis()
plt.show()

Почему пропуски возникли и что с ними делать — рассуждение о предметной области, а не только техника:

| Столбец | Пропусков | Почему, вероятно | Решение |
|---|---|---|---|
| `Cabin` | 77 % | номера кают сохранились в основном для 1-го класса; у 3-го класса отдельной каюты часто не было | столбец удалить, но **факт наличия записи** оставить как признак `HasCabin` — сам пропуск информативен |
| `Age` | 20 % | возраст восстанавливали по архивам, не для всех | заполнить медианой **внутри группы** (по титулу и классу — у `Master` и `Mr` разный возраст) и добавить флаг `AgeMissing` |
| `Embarked` | 2 строки | ошибки записи | заполнить самым частым значением (`S`) |

Общее правило: сначала понять механизм пропуска, потом выбирать между *удалить строку / удалить столбец / заполнить константой / заполнить статистикой / заполнить по группе / добавить индикатор*.

In [ ]:
has_cabin = df['Cabin'].notna()
print("доля записей о каюте по классам:\n", has_cabin.groupby(df['Pclass']).mean().round(2), sep='')
print("\nдоля выживших среди тех, у кого каюта (не)известна:\n", df['Survived'].groupby(has_cabin).mean().round(2), sep='')

Гипотеза подтвердилась: запись о каюте есть у 81 % пассажиров 1-го класса и у 2 % третьего, а выживаемость «с каютой» вдвое выше. Пропуск — это информация.

### 5.4 Распределения признаков

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(df['Age'].dropna(), bins=30, color=BLUE, edgecolor='white')
axes[0].set_title('Age'); axes[0].set_xlabel('лет'); axes[0].set_ylabel('пассажиров')
axes[1].hist(df['Fare'], bins=50, color=BLUE, edgecolor='white')
axes[1].set_title('Fare'); axes[1].set_xlabel('£')
plt.tight_layout(); plt.show()

print(df[['Age', 'Fare']].describe().round(1))
print("асимметрия (skew): Age =", round(df['Age'].skew(), 2), "| Fare =", round(df['Fare'].skew(), 2))

**Наблюдения.**
- `Age`: почти симметричное распределение с модой 20–30 лет и заметным «горбиком» младенцев (0–5 лет). Медиана 28, среднее 29.7 — близко, выбросов мало.
- `Fare`: сильно **скошено вправо** (skew ≈ 4.8): половина билетов дешевле 14.5 £, но хвост тянется до 512 £. Среднее (32) заметно больше медианы (14.5) — классический признак скошенности. Для линейных моделей такой признак обычно логарифмируют.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(df['Fare'], bins=50, color=BLUE, edgecolor='white')
axes[0].set_title('Fare — как есть')
axes[1].hist(np.log1p(df['Fare']), bins=50, color=BLUE, edgecolor='white')
axes[1].set_title('log(1 + Fare) — хвост «подтянут»')
for ax in axes:
    ax.set_ylabel('пассажиров')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, col in zip(axes, ['Pclass', 'Sex', 'Embarked', 'SibSp']):
    counts = df[col].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color=BLUE, width=0.6)
    ax.set_title(col)
axes[0].set_ylabel('пассажиров')
plt.tight_layout(); plt.show()

Третий класс — больше половины пассажиров, мужчин 65 %, из Саутгемптона 72 %, большинство без братьев/сестёр/супругов. Это важно помнить при чтении следующих графиков: доля выживших в группе из 15 человек — шумная величина.

In [ ]:
df.sort_values('Fare', ascending=False)[['Name', 'Pclass', 'Fare', 'Ticket']].head(5)    # выбросы: кто это?

Три пассажира с билетом за 512 £ ехали по **одному** билету `PC 17755` — это не ошибка данных, а реальность (семья с прислугой; стоимость всей группы записана каждому). Выбросы не удаляют «потому что мешают» — сначала выясняют их природу. Для линейных моделей их влияние смягчит логарифм, деревьям они не мешают.

### 5.5 Как признаки связаны с целевой переменной

Главный приём: **сгруппировать по признаку и посчитать долю выживших**. Если доля сильно отличается между группами — признак информативен для модели. Пунктирная линия на графиках — средняя выживаемость по всем (38 %).

In [ ]:
def survival_bar(col, ax=None, title=None):
    '''Доля выживших по значениям столбца col; над столбиком — доля и размер группы.'''
    g = df.groupby(col, observed=True)['Survived'].agg(['mean', 'count'])
    ax = ax or plt.gca()
    bars = ax.bar(g.index.astype(str), g['mean'], color=BLUE, width=0.6)
    ax.bar_label(bars, labels=[f"{m:.0%}\n(n={n})" for m, n in zip(g['mean'], g['count'])], padding=2, fontsize=8)
    ax.axhline(df['Survived'].mean(), color=GRAY, linestyle=':', linewidth=1)   # средняя выживаемость — ориентир
    ax.set_ylim(0, 1.2)
    ax.set_title(title or f'Доля выживших по {col}')
    return ax

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
survival_bar('Sex', axes[0])
survival_bar('Pclass', axes[1])
survival_bar('Embarked', axes[2])
axes[0].set_ylabel('доля выживших')
plt.tight_layout(); plt.show()

- **`Sex`** — самый сильный сигнал: выжили 74 % женщин и 19 % мужчин. «Женщины и дети вперёд» видно в данных напрямую.
- **`Pclass`** — монотонно: 63 % → 47 % → 24 %.
- **`Embarked`** — пассажиры из Шербура (C) выживали чаще. Порт сам по себе не спасает: проверим, не прячется ли за ним класс билета.

In [ ]:
pd.crosstab(df['Embarked'], df['Pclass'], normalize='index').round(2)    # доля классов внутри каждого порта

Из Шербура 51 % пассажиров ехали 1-м классом, из Куинстауна 94 % — третьим. Эффект порта почти целиком объясняется классом. Это **скрытая переменная (confounder)**: корреляция есть, причинности нет. Такие вещи ловят только сводными таблицами по двум признакам.

In [ ]:
bins = np.arange(0, 85, 5)
age_bins = pd.cut(df['Age'], bins=bins, right=False)
rate_by_age = df.groupby(age_bins, observed=True)['Survived'].agg(['mean', 'count'])
centers = [iv.mid for iv in rate_by_age.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

ax = axes[0]
ax.hist([df.loc[df['Survived'] == 0, 'Age'].dropna(), df.loc[df['Survived'] == 1, 'Age'].dropna()],
        bins=bins, stacked=True, color=[BLUE, ORANGE], label=['погиб', 'выжил'], edgecolor='white', linewidth=0.5)
ax.set_xlabel('возраст, лет'); ax.set_ylabel('пассажиров'); ax.set_title('Возраст в разрезе выживаемости'); ax.legend()

ax = axes[1]
ax.plot(centers, rate_by_age['mean'], marker='o', color=BLUE, linewidth=2)
ax.axhline(df['Survived'].mean(), color=GRAY, linestyle=':', linewidth=1)
ax.set_ylim(0, 1); ax.set_xlabel('возраст, лет (интервалы по 5)'); ax.set_ylabel('доля выживших')
ax.set_title('Доля выживших по возрасту')
plt.tight_layout(); plt.show()

Дети до 10 лет выживали чаще среднего («дети вперёд»), 20–30 лет — реже всего (много мужчин 3-го класса), после 60 — плохо. Зависимость **нелинейная** — запомним это для раздела с корреляциями. Правый край графика (70+) — это 1–5 человек на интервал, доверять ему нельзя.

In [ ]:
fare_q = pd.qcut(df['Fare'], q=4, labels=['дешёвые 25 %', '25–50 %', '50–75 %', 'дорогие 25 %'])   # квартили
rate_by_fare = df.groupby(fare_q, observed=True)['Survived'].mean()

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(rate_by_fare.index.astype(str), rate_by_fare.values, color=BLUE, width=0.6)
ax.bar_label(bars, fmt='%.2f', padding=2)
ax.axhline(df['Survived'].mean(), color=GRAY, linestyle=':', linewidth=1)
ax.set_ylim(0, 1); ax.set_title('Доля выживших по квартилям цены билета'); ax.set_ylabel('доля выживших')
plt.show()

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

fig, ax = plt.subplots(figsize=(7.5, 3.5))
survival_bar('FamilySize', ax, title='Доля выживших по размеру семьи на борту')
ax.set_ylabel('доля выживших'); ax.set_xlabel('FamilySize = SibSp + Parch + 1')
plt.show()

Одиночки — 30 %, семьи из 2–4 человек — 55–72 %, большие семьи (5+) — резко вниз. Зависимость опять нелинейная: линейная модель её не поймает, дерево — поймает. Отсюда идея бинарного признака `IsAlone` и самого `FamilySize` — ни тот, ни другой в исходных данных не было.

### 5.6 Два признака одновременно

In [ ]:
pivot = pd.pivot_table(df, values='Survived', index='Pclass', columns='Sex', aggfunc='mean')

fig, ax = plt.subplots(figsize=(4.5, 3.5))
im = ax.imshow(pivot.values, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(pivot.columns)), pivot.columns)
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.set_xlabel('Sex'); ax.set_ylabel('Pclass'); ax.set_title('Доля выживших: класс × пол')
ax.grid(False)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha='center', va='center', color='white' if v > 0.5 else 'black')
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()
# То же самое одной строкой с seaborn: sns.heatmap(pivot, annot=True, cmap='Blues', vmin=0, vmax=1)

Женщины 1–2 класса — 92–97 %; мужчины 2–3 класса — 13–16 %. Пол и класс вместе почти «решают» задачу — отсюда и простой бейзлайн, который построим в конце.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for value, color, label in [(0, BLUE, 'погиб'), (1, ORANGE, 'выжил')]:
    part = df[df['Survived'] == value]
    ax.scatter(part['Age'], part['Fare'], s=14, alpha=0.6, color=color, label=label,
               edgecolors='white', linewidths=0.3)
ax.set_yscale('log')                         # цена скошена — логарифмическая ось
ax.set_xlabel('возраст, лет'); ax.set_ylabel('цена билета, £ (лог. шкала)')
ax.set_title('Возраст × цена билета в разрезе выживаемости'); ax.legend()
plt.show()

Верхняя часть облака (дорогие билеты) заметно «оранжевее», нижняя — «синее»; слева внизу (дети с дешёвыми билетами) оранжевых тоже много. Два числовых признака + цвет класса — стандартный способ увидеть, разделимы ли классы вообще.

### 5.7 Корреляции

In [ ]:
corr_df = df.assign(SexCode=df['Sex'].map({'male': 0, 'female': 1}))
corr = corr_df[['Survived', 'SexCode', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']].corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr)), corr.index)
ax.grid(False)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha='center', va='center', fontsize=8, color='white' if abs(v) > 0.5 else 'black')
fig.colorbar(im, ax=ax, shrink=0.8, label='коэффициент корреляции Пирсона')
ax.set_title('Матрица корреляций')
plt.tight_layout(); plt.show()

print(corr['Survived'].drop('Survived').sort_values().round(3))

**Интерпретация.** Корреляция Пирсона измеряет *линейную* связь, от −1 до 1.
- `SexCode` (+0.54) и `Pclass` (−0.34) — самые сильные связи с целевой: женщины и высокий класс (меньший номер) выживали чаще.
- `Fare` (+0.26) частично дублирует `Pclass` (между ними −0.55). Это мультиколлинеарность: деревьям безразлично, у линейных моделей портит интерпретацию весов.
- `Age` (−0.08) почти не коррелирует с целевой, хотя график показал явную зависимость для детей. **Слабая корреляция ≠ бесполезный признак**: связь U-образная, Пирсон её не видит.
- Корреляция ≠ причинность: `Embarked = C` «повышал» выживаемость только потому, что из Шербура ехало больше 1-го класса.

## 6. Инженерия признаков: превращаем наблюдения в столбцы

Всё, что мы заметили в EDA, нужно **записать в таблицу числами** — модель работает только с числовой матрицей. Этот этап называется feature engineering, и на табличных данных именно он, а не выбор алгоритма, чаще всего определяет качество.

| Наблюдение из EDA | Признак |
|---|---|
| семьи среднего размера выживали чаще, одиночки — реже | `FamilySize`, `IsAlone` |
| титул в имени кодирует пол, возраст и статус | `Title` из `Name` |
| пропуск `Cabin` сам по себе информативен | `HasCabin` |
| возраст пропущен у 20 % и зависит от титула и класса | `Age` заполняем медианой по группе + флаг `AgeMissing` |
| `Fare` скошен | `LogFare` |
| `Sex`, `Embarked` — текст | кодирование 0/1 и one-hot |

In [ ]:
feat = df.copy()      # работаем с копией, исходную таблицу не трогаем

feat['Title'] = feat['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
print(feat['Title'].value_counts())

Регулярное выражение `r',\s*([^\.]+)\.'` читается так: «после запятой пропусти пробелы и захвати всё до первой точки».

| Элемент | Что делает |
|---|---|
| `,` | буквальная запятая после фамилии: `Braund, Mr. Owen Harris` |
| `\s*` | ноль или больше пробельных символов |
| `([^\.]+)` | захватывающая группа: один и более символов, **не являющихся точкой** |
| `\.` | буквальная точка, завершающая титул |

Редкие титулы (`Dr`, `Rev`, `Col`, …) сведём в `Rare`, синонимы (`Mlle` → `Miss`, `Mme` → `Mrs`) объединим — иначе получим категории из 1–2 человек, по которым модель ничему не научится.

In [ ]:
feat['Title'] = feat['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
common = ['Mr', 'Miss', 'Mrs', 'Master']
feat['Title'] = feat['Title'].where(feat['Title'].isin(common), 'Rare')

print(feat.groupby('Title')['Survived'].agg(['mean', 'count']).round(2))
print("\nмедианный возраст по титулу:\n", feat.groupby('Title')['Age'].median(), sep='')   # Master — дети, Mr — взрослые

`Master` — это мальчики (медиана 3.5 года), `Mr` — взрослые мужчины (30). Заполнять пропуски возраста одной общей медианой (28) значило бы записать мальчикам по 28 лет. Поэтому заполняем **внутри группы** `Title × Pclass`.

In [ ]:
feat['FamilySize'] = feat['SibSp'] + feat['Parch'] + 1
feat['IsAlone'] = (feat['FamilySize'] == 1).astype(int)
feat['HasCabin'] = feat['Cabin'].notna().astype(int)
feat['AgeMissing'] = feat['Age'].isna().astype(int)

# Возраст: медиана внутри группы. transform возвращает Series той же длины, выровненную по индексу исходной таблицы
feat['Age'] = feat.groupby(['Title', 'Pclass'])['Age'].transform(lambda s: s.fillna(s.median()))
feat['Age'] = feat['Age'].fillna(feat['Age'].median())      # страховка на случай группы, где возраст не известен ни у кого

feat['Embarked'] = feat['Embarked'].fillna(feat['Embarked'].mode()[0])   # самое частое значение
feat['LogFare'] = np.log1p(feat['Fare'])

print(feat.isna().sum()[lambda s: s > 0])    # пропуски остались только в Cabin — его мы удалим

### Кодирование категорий

Модели нужны числа. `Sex` — две категории, хватит 0/1. `Embarked` и `Title` — больше двух, и присвоить им 0, 1, 2, 3 нельзя: модель решит, что `Q` «больше» `C`, а `Mrs` — «среднее между `Miss` и `Rare`». Стандартный приём — **one-hot**: по бинарному столбцу-индикатору на каждую категорию (`pd.get_dummies`, в sklearn — `OneHotEncoder`).

In [ ]:
feat['SexCode'] = feat['Sex'].map({'male': 0, 'female': 1})
feat = pd.get_dummies(feat, columns=['Embarked', 'Title'], dtype=int)

feature_cols = (['Pclass', 'SexCode', 'Age', 'AgeMissing', 'SibSp', 'Parch', 'FamilySize', 'IsAlone', 'LogFare', 'HasCabin']
                + [c for c in feat.columns if c.startswith(('Embarked_', 'Title_'))])
X = feat[feature_cols]
y = feat['Survived']
print("X:", X.shape, "| y:", y.shape)
X.head()

> **Важно для ЛР3.** Медианы и моды здесь посчитаны по *всем* данным — для разведки это допустимо. При обучении модели статистики для заполнения пропусков и масштабирования считают **только на обучающей части**, а к тестовой применяют уже готовыми. Иначе информация из теста «протекает» в обучение (*data leakage*), и оценка качества получается завышенной. В scikit-learn для этого есть `Pipeline` и `ColumnTransformer`.

## 7. Первая модель: зачем всё это было

У нас есть числовая матрица `X` и вектор `y` — ровно то, что ждёт любая ML-библиотека. Построим «модели» по возрастанию сложности и сравним их честно — на отложенной части данных, которую модель не видела при обучении.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("обучение:", X_train.shape, "| проверка:", X_valid.shape)
print("доля выживших — train:", round(y_train.mean(), 3), "| valid:", round(y_valid.mean(), 3))   # stratify сохранил пропорцию

In [ ]:
y_true = y_valid.to_numpy()

# Бейзлайн 0: «все погибли» — accuracy равна доле класса большинства
pred_majority = np.zeros(len(y_true), dtype=int)

# Бейзлайн 1: правило из EDA — «выживают женщины»
pred_rule = (X_valid['SexCode'] == 1).astype(int).to_numpy()

print(f"все погибли:        accuracy = {(pred_majority == y_true).mean():.3f}")
print(f"правило «женщины»:  accuracy = {(pred_rule == y_true).mean():.3f}")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))   # масштабирование + линейная модель
forest = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)

for name, model in [('логистическая регрессия', logreg), ('случайный лес', forest)]:
    model.fit(X_train, y_train)                       # обучение — одна строка
    print(f"{name:24s} accuracy: train = {accuracy_score(y_train, model.predict(X_train)):.3f}, "
          f"valid = {accuracy_score(y_valid, model.predict(X_valid)):.3f}")

In [ ]:
cm = confusion_matrix(y_valid, forest.predict(X_valid))
print(pd.DataFrame(cm, index=['истина: погиб', 'истина: выжил'], columns=['предсказано: погиб', 'предсказано: выжил']))

importances = pd.Series(forest.feature_importances_, index=feature_cols).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(importances.index, importances.values, color=BLUE, height=0.6)
ax.set_title('Важность признаков (случайный лес)'); ax.set_xlabel('доля вклада')
plt.show()

**Что мы увидели.**
- Простое правило из EDA («женщины выживают») даёт около 0.78, а обученные модели превосходят его лишь на 5–7 процентных пунктов. Это нормально: **хороший бейзлайн — половина работы**, и его строят первым, чтобы было с чем сравнивать.
- Важность признаков повторяет выводы EDA: `SexCode` / `Title_Mr`, `LogFare`, `Pclass`, `Age`. Признаки, которые мы *сконструировали* (`Title_*`, `LogFare`, `HasCabin`, `FamilySize`), — в верхней половине списка. Вот зачем был раздел 6.
- У случайного леса accuracy на train заметно выше, чем на valid, — признак переобучения; у логистической регрессии разрыва нет. Как это измерять и лечить, какие ещё есть метрики (precision, recall, ROC-AUC) и модели — ЛР3.
- Те же `X` и `y` потом пойдут в нейросеть: `torch.tensor(X.to_numpy(), dtype=torch.float32)` — данные те же, меняется только контейнер (ЛР4–5).

## 8. Итоги

**Три уровня данных:** списки Python → массивы NumPy (скорость, математика) → таблицы pandas (имена, типы, пропуски). Модель на входе ждёт NumPy-матрицу `(n_samples, n_features)`.

**Главные привычки**
1. Векторизуйте: цикл по элементам — почти всегда ошибка.
2. Смотрите данные глазами: `head / sample / info / describe` — до любых графиков.
3. Пропуск — это информация. Сначала «почему», потом «чем заполнить».
4. Группировка по признаку + доля целевого класса — основной инструмент EDA.
5. Корреляция видит только линейное; график видит всё. Сводная таблица по двум признакам ловит скрытые переменные.
6. Сначала бейзлайн, потом модель.

### Чеклист ЛР2 на вашем датасете

| № | Пункт ЛР2 | Инструмент из лекции |
|---|---|---|
| 2 | постановка: регрессия или классификация, что за переменные | словарь данных, `dtypes`, `nunique()` |
| 3 | распределение целевой переменной, дисбаланс | `value_counts(normalize=True)`, `plt.bar` / `plt.hist` |
| 4 | основные статистики | `describe()` |
| 5 | информация о столбцах | `info()` |
| 6 | пропуски и гипотезы о причинах | `isna().sum()`, `isna().mean()`, таблица «почему → что делать» |
| 7 | гистограммы 5 признаков, скошенность | `plt.hist`, `skew()`, `np.log1p` |
| 8 | связь 5 признаков с целевой | `groupby().mean()`, `pd.cut`, `pd.qcut`, графики |
| 9 | два признака сразу (extra) | `pivot_table` + heatmap, `scatter` с цветом по классу |

Для регрессии (непрерывная целевая) пункт 8 — это `scatter(x, y)` и `groupby(pd.cut(x))['y'].mean()` вместо доли класса.

### Документация
- NumPy для начинающих: https://numpy.org/doc/stable/user/absolute_beginners.html
- pandas за 10 минут: https://pandas.pydata.org/docs/user_guide/10min.html · шпаргалка: https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf
- matplotlib pyplot: https://matplotlib.org/stable/tutorials/pyplot.html
- scikit-learn, первые шаги: https://scikit-learn.org/stable/getting_started.html
- Соревнование Kaggle Titanic: https://www.kaggle.com/c/titanic

## 9. Мини-задания для самопроверки

Заполните функции так, чтобы прошли `assert`'ы (как в ЛР1). Каждая решается в 1–3 строки **без циклов**. `df` — чистая таблица Titanic из раздела 5 (плюс столбец `FamilySize`).

In [ ]:
def standardize(M):
    '''Вернуть копию матрицы M формы (n_samples, n_features), в которой каждый столбец имеет среднее 0 и std 1.'''
    # TODO: напиши меня
    pass


M = np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0], [4.0, 40.0]])
S = standardize(M)
assert S.shape == M.shape
assert np.allclose(S.mean(axis=0), 0) and np.allclose(S.std(axis=0), 1)

In [ ]:
def accuracy(y_true, y_pred):
    '''Доля совпадений двух массивов из 0/1. Без циклов.'''
    # TODO: напиши меня
    pass


assert accuracy(np.array([1, 0, 1, 1]), np.array([1, 0, 0, 1])) == 0.75
assert accuracy(np.array([0, 0]), np.array([1, 1])) == 0.0

In [ ]:
def children_survival_rate(data, max_age=12):
    '''Доля выживших среди пассажиров, чей возраст известен и строго меньше max_age.'''
    # TODO: напиши меня
    pass


assert round(children_survival_rate(df), 3) == 0.574
assert round(children_survival_rate(df, max_age=18), 3) == 0.540

In [ ]:
def survival_by_port(data):
    '''Series: индекс — порт посадки (Embarked), значения — доля выживших, по убыванию. Строки без порта не учитывать.'''
    # TODO: напиши меня
    pass


res = survival_by_port(df)
assert list(res.index) == ['C', 'Q', 'S']
assert round(res['C'], 3) == 0.554

In [ ]:
def most_common_surnames(data, n=2):
    '''Список из n самых частых фамилий (часть Name до запятой) по убыванию частоты.'''
    # TODO: напиши меня
    pass


assert most_common_surnames(df) == ['Andersson', 'Sage']